神经网络可以使用 torch.nn 包来构建。

既然已经初步了解了 autograd，那么 nn 就依赖于 autograd 来定义模型并对其求导。nn.Module 包含各个层，以及一个返回 output 的 forward(input) 方法。

神经网络的典型训练过程如下：

定义具有可学习参数（或权重）的神经网络

在输入 数据集上进行迭代

通过网络处理输入

计算损失（输出距离目标有多远）

将梯度反向传播回网络的参数中

更新网络的权重，通常使用简单的更新规则：权重 = 权重 - 学习率 * 梯度

# 定义网络

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):

    def __init__(self):
        super().__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        # kernel
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # an affine operation: y = Wx + b
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # 5*5 from image dimension
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, input):
        # Convolution layer C1: 1 input image channel, 6 output channels,
        # 5x5 square convolution, it uses RELU activation function, and
        # outputs a Tensor with size (N, 6, 28, 28), where N is the size of the batch
        c1 = F.relu(self.conv1(input))
        # Subsampling layer S2: 2x2 grid, purely functional,
        # this layer does not have any parameter, and outputs a (N, 6, 14, 14) Tensor
        s2 = F.max_pool2d(c1, (2, 2))
        # Convolution layer C3: 6 input channels, 16 output channels,
        # 5x5 square convolution, it uses RELU activation function, and
        # outputs a (N, 16, 10, 10) Tensor
        c3 = F.relu(self.conv2(s2))
        # Subsampling layer S4: 2x2 grid, purely functional,
        # this layer does not have any parameter, and outputs a (N, 16, 5, 5) Tensor
        s4 = F.max_pool2d(c3, 2)
        # Flatten operation: purely functional, outputs a (N, 400) Tensor
        s4 = torch.flatten(s4, 1)
        # Fully connected layer F5: (N, 400) Tensor input,
        # and outputs a (N, 120) Tensor, it uses RELU activation function
        f5 = F.relu(self.fc1(s4))
        # Fully connected layer F6: (N, 120) Tensor input,
        # and outputs a (N, 84) Tensor, it uses RELU activation function
        f6 = F.relu(self.fc2(f5))
        # Fully connected layer OUTPUT: (N, 84) Tensor input, and
        # outputs a (N, 10) Tensor
        output = self.fc3(f6)
        return output


net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


模型的可学习参数由 net.parameters() 返回

In [2]:
params = list(net.parameters())
print(len(params))
print(params[0].size())  # conv1's .weight

10
torch.Size([6, 1, 5, 5])


让我们尝试一个随机的 32x32 输入。注意：此网络（LeNet）预期的输入尺寸为 32x32。要在 MNIST  数据集上使用此网络，请将数据集中的图像调整为 32x32。

In [3]:
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

tensor([[ 0.0370,  0.0099, -0.0182, -0.0214, -0.0470,  0.1024, -0.0505,  0.0768,
          0.0234,  0.0377]], grad_fn=<AddmmBackward0>)


将所有参数的梯度缓冲区置零，并使用随机梯度进行反向传播

In [4]:
net.zero_grad()
out.backward(torch.randn(1, 10))

在继续深入之前，让我们回顾一下目前为止所见的所有类。

回顾
torch.Tensor - 一个支持自动求导操作（如 backward()）的多维数组。同时保存有关该张量的梯度。

nn.Module - 神经网络模块。封装参数的便捷方式，并提供将参数移动到 GPU、导出、加载等的辅助方法。

nn.Parameter - 一种张量，当它被分配为 Module 的属性时，会被自动注册为参数。

autograd.Function - 实现自动求导操作的前向和反向定义。每个 Tensor 操作至少创建一个 Function 节点，该节点连接到创建 Tensor 的函数，并对其历史记录进行编码。

目前为止，我们已经涵盖了：
定义神经网络

处理输入并调用反向传播

尚未学习：
计算损失

更新网络权重

# 损失函数
损失函数接收 (output, target) 对作为输入，并计算一个值来评估输出距离目标有多远。

nn 包下有几种不同的 损失函数。一种简单的损失是：nn.MSELoss，它计算输出和目标之间的均方误差。

In [5]:
output = net(input)
target = torch.randn(10)  # a dummy target, for example
target = target.view(1, -1)  # make it the same shape as output
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(0.6617, grad_fn=<MseLossBackward0>)


input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
      -> flatten -> linear -> relu -> linear -> relu -> linear
      -> MSELoss
      -> loss

因此，当我们调用 loss.backward() 时，整个图都会相对于神经网络参数进行求导，图中所有 requires_grad=True 的张量都将累积其梯度到 .grad 张量中。机器学习与人工智能

为了说明，让我们跟踪几个反向步骤

In [6]:
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

# 反向传播
要反向传播误差，我们所要做的就是 loss.backward()。不过你需要清除现有的梯度，否则梯度会累积到现有的梯度上。

现在我们将调用 loss.backward()，并观察 conv1 的偏置梯度在反向传播前后的变化。由于我们还没有引入优化器，我们直接在模型上清除梯度。一旦使用优化器，建议使用如下所示的 optimizer.zero_grad()。

In [7]:
net.zero_grad()     # zeroes the gradient buffers of all parameters

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([-0.0087,  0.0015, -0.0021, -0.0048,  0.0040, -0.0153])


# 更新权重
实践中使用的最简单的更新规则是随机梯度下降 (SGD)

weight = weight - learning_rate * gradient

In [9]:
learning_rate = 0.01
for f in net.parameters():
    with torch.no_grad():
        f -= f.grad * learning_rate

然而，当你在使用神经网络时，你会想要使用各种不同的更新规则，例如 SGD、Nesterov-SGD、Adam、RMSProp 等。为了实现这一点，我们构建了一个小包：torch.optim，它实现了所有这些方法。使用它非常简单：

In [10]:
import torch.optim as optim

# create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()    # Does the update